# 03 · Business baseline (§8.7)

Historical call-center metrics: FCR (`was_resolved` without `requires_followup`), escalation, duration/wait and CSAT.
It is the bar the system is **projected** against: labeled as a projection (CON-07), never as a measured improvement.

Output: `analysis/results/business_baseline.json`.

In [ ]:
from common import connect, save_result, show

con = connect()

In [ ]:
baseline = show(con, "Call center by category (CSAT: CSAT surveys, 1-4 scale, joined by interaction_id)", """
    SELECT i.reason_category, count(*) AS n,
           round(avg((i.was_resolved AND NOT i.requires_followup)::INT), 3) AS fcr,
           round(avg(i.was_escalated::INT), 3) AS escalacion,
           round(median(i.duration_seconds)) AS dur_p50_s, round(median(i.wait_time_seconds)) AS espera_p50_s,
           round(avg(s.main_score), 2) AS csat, count(s.main_score) AS csat_n
    FROM call_center_interactions i
    LEFT JOIN (SELECT interaction_id, main_score FROM satisfaction_surveys WHERE survey_type = 'CSAT') s
           USING (interaction_id)
    GROUP BY 1 ORDER BY n DESC""")


In [ ]:
# Scope: disputes arrive as complaint contacts ("Queja"; contact_reason only repeats the category, and dispute
# complaints carry no origin_interaction_id that joins, 0/24,491). All contacts are kept as the reference line.
scope = baseline.filter(baseline["reason_category"] == "Queja").to_dicts()[0]
total = show(con, "All contacts (reference)", """
    SELECT count(*) AS n, round(avg((i.was_resolved AND NOT i.requires_followup)::INT), 3) AS fcr,
           round(avg(i.was_escalated::INT), 3) AS escalacion,
           round(median(i.duration_seconds)) AS dur_p50_s, round(median(i.wait_time_seconds)) AS espera_p50_s,
           round(avg(s.main_score), 2) AS csat
    FROM call_center_interactions i
    LEFT JOIN (SELECT interaction_id, main_score FROM satisfaction_surveys WHERE survey_type = 'CSAT') s
           USING (interaction_id)""").to_dicts()[0]
scope


In [ ]:
save_result("business_baseline", {
    "label": "historical",
    "scope": "complaint contacts (reason_category = Queja)",
    "n": scope["n"],
    "fcr": scope["fcr"],
    "escalacion": scope["escalacion"],
    "dur_p50_s": scope["dur_p50_s"],
    "espera_p50_s": scope["espera_p50_s"],
    "csat": f"{scope['csat']} / 4 (n={scope['csat_n']:,})",
    "all_contacts": total,
    "by_reason": baseline.to_dicts(),
})
